<a href="https://colab.research.google.com/github/rizqihilman/rizqihilman/blob/main/Data_Cleansing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load Dataset

In [ ]:
# change the path according to your file location in your Google Drive
df = pd.read_csv('/content/drive/MyDrive/botak.csv')

In [ ]:
# informasi general tentang dataframe
df.info()

Pengamatan\:
1. Data terdiri dari 7991 baris
2. Tampak beberapa kolom masih memiliki null/missing values (Non-Null Count < jumlah baris)
3. Sepertinya tidak ada issue yang mencolok pada tipe data untuk setiap kolom (sudah sesuai)

## Cuplikan Data

In [ ]:
# 5 baris teratas dari data
df.head()

In [ ]:
# pengelompokan kolom berdasarkan jenisnya
nums = ['umur', 'gaji', 'is_menikah', 'is_keturunan', 'berat', 'tinggi', 'is_merokok', 'botak_prob']
cats = ['jenis_kelamin', 'pekerjaan', 'sampo', 'pendidikan','provinsi']

# Missing Data

In [ ]:
# jumlah entry NULL di setiap kolom
df.isna().sum()

In [ ]:
# drop baris dengan missing values
df = df.dropna()

In [ ]:
# cek apakah sudah berhasil
df.isna().sum()

# Duplicated Data

In [ ]:
# cek jumlah duplicated rows
# dari semua kolom
df.duplicated().sum()

In [ ]:
# drop duplicated rows
print(df.duplicated().sum())
df.drop_duplicates(inplace=True)
print(df.duplicated().sum())

# Outliers Removal
Menggunakan konsep IQR

In [ ]:
print(f'Jumlah baris sebelum memfilter outlier: {len(df)}')

for col in ['umur','gaji', 'berat', 'tinggi']:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3-Q1
  low_limit = Q1 - 1.5*IQR
  high_limit = Q3 + 1.5*IQR
  df = df[(df[col] >= low_limit) & (df[col] <= high_limit)]

print(f'Jumlah baris setelah memfilter outlier: {len(df)}')

# Feature Encoding

In [ ]:
# pengecekan nilai/entri dari kolom-kolom kategorikal

for col in cats:
  print(f'value counts of column {col}')
  print(df[col].value_counts())
  print('---'*10, '\n')

Strategi encoding
* `jenis_kelamin` & `pendidikan` \: label encoding
* `pekerjaan` & `sampo` \: One Hot Encoding 
* `provinsi`: sepertinya dapat di-drop, karena distinct values terlalu banyak

In [ ]:
# drop kolom provinsi
df = df.drop(columns='provinsi')

In [ ]:
# label encoding
# untuk kolom jenis_kelamin dan pendidikan
mapping_jenis_kelamin = {
    'Perempuan' : 0,
    'Laki-laki' : 1
}

mapping_pendidikan = {
    'SD' : 0,
    'SMP' : 1,
    'SMA' : 2,
    'S1' : 3,
    'S2' : 4,
    'S3' : 5
}

df['jenis_kelamin'] = df['jenis_kelamin'].map(mapping_jenis_kelamin)
df['pendidikan'] = df['pendidikan'].map(mapping_pendidikan)

In [ ]:
# One hot encoder pekerjaan dan sampo
# WARNING: run code ini sekali saja
for cat in ['pekerjaan', 'sampo']:
    onehots = pd.get_dummies(df[cat], prefix=cat)
    df = df.join(onehots)

# Drop kolom-kolom yang outdated

In [ ]:
# drop kolom pekerjaan dan sampo yang asli (karena sudah di-encoding)
df = df.drop(columns=['pekerjaan', 'sampo'])

In [ ]:
# cek 'n ricek penampakan data setelah preprocessing
df.head()

In [ ]:
# informasi dataframe yang sudah di-preprocessing
df.info()